In [ ]:
# ============================================================
# INDIAN SIGN LANGUAGE (ISL) SENTENCE CLASSIFICATION
# COMPLETE TRAINING PIPELINE
# MediaPipe Tasks API Version
# Optimized for Mac M1 / CPU
# ============================================================

# INSTALL FIRST:
# pip install mediapipe opencv-python tensorflow scikit-learn tqdm

from pathlib import Path
import cv2
import numpy as np
from tqdm import tqdm
import mediapipe as mp

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Dense,
    Dropout,
    Bidirectional,
    BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# ============================================================
# DATASET PATH
# ============================================================

DATA_DIR = Path(
    "isl_nmf/data/isl_csltr/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus/Videos_Sentence_Level"
)

# ============================================================
# CONFIG
# ============================================================

SEQ_LEN = 30

# pose (33*3) + left hand (21*3) + right hand (21*3)
NUM_KEYPOINTS = 225

FRAME_SKIP = 5

# ============================================================
# MEDIAPIPE TASKS API
# ============================================================

BaseOptions = mp.tasks.BaseOptions

VisionRunningMode = mp.tasks.vision.RunningMode

HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions

# ============================================================
# MODEL FILES
# ============================================================

# DOWNLOAD THESE FILES:
#
# hand_landmarker.task
# https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
#
# pose_landmarker_full.task
# https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task
#
# Put both files in SAME folder as this script/notebook.

HAND_MODEL = "hand_landmarker.task"
POSE_MODEL = "pose_landmarker_full.task"

# ============================================================
# LOAD MEDIAPIPE MODELS
# ============================================================

hand_options = HandLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path=HAND_MODEL
    ),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=2
)

pose_options = PoseLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path=POSE_MODEL
    ),
    running_mode=VisionRunningMode.IMAGE
)

hand_detector = HandLandmarker.create_from_options(
    hand_options
)

pose_detector = PoseLandmarker.create_from_options(
    pose_options
)

print("✓ MediaPipe Tasks models loaded")

# ============================================================
# LOAD CLASS NAMES
# ============================================================

class_names = sorted([
    d.name
    for d in DATA_DIR.iterdir()
    if d.is_dir()
])

print(f"\nTotal Classes: {len(class_names)}")
print(class_names[:10])

# ============================================================
# EXTRACT KEYPOINTS
# ============================================================

def extract_keypoints(frame):

    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    # ========================================================
    # POSE
    # ========================================================

    pose_result = pose_detector.detect(mp_image)

    pose = np.zeros(33 * 3)

    if pose_result.pose_landmarks:

        pose_landmarks = pose_result.pose_landmarks[0]

        pose = np.array([
            [lm.x, lm.y, lm.z]
            for lm in pose_landmarks
        ]).flatten()

    # ========================================================
    # HANDS
    # ========================================================

    hand_result = hand_detector.detect(mp_image)

    left_hand = np.zeros(21 * 3)
    right_hand = np.zeros(21 * 3)

    if hand_result.hand_landmarks:

        for idx, hand_landmarks in enumerate(
            hand_result.hand_landmarks
        ):

            handedness = (
                hand_result.handedness[idx][0]
                .category_name
            )

            hand_array = np.array([
                [lm.x, lm.y, lm.z]
                for lm in hand_landmarks
            ]).flatten()

            if handedness == "Left":
                left_hand = hand_array
            else:
                right_hand = hand_array

    # ========================================================
    # CONCATENATE
    # ========================================================

    keypoints = np.concatenate([
        pose,
        left_hand,
        right_hand
    ])

    return keypoints

# ============================================================
# LOAD VIDEO
# ============================================================

def load_video(video_path):

    cap = cv2.VideoCapture(str(video_path))

    frames = []

    frame_idx = 0

    while cap.isOpened():

        ret, frame = cap.read()

        if not ret:
            break

        frame_idx += 1

        # ====================================================
        # FRAME SKIPPING
        # ====================================================

        if frame_idx % FRAME_SKIP != 0:
            continue

        keypoints = extract_keypoints(frame)

        frames.append(keypoints)

    cap.release()

    # ========================================================
    # EMPTY VIDEO
    # ========================================================

    if len(frames) == 0:
        return None

    frames = np.array(frames)

    # ========================================================
    # FIX SEQUENCE LENGTH
    # ========================================================

    if len(frames) > SEQ_LEN:

        idx = np.linspace(
            0,
            len(frames) - 1,
            SEQ_LEN
        ).astype(int)

        frames = frames[idx]

    elif len(frames) < SEQ_LEN:

        padding = np.zeros((
            SEQ_LEN - len(frames),
            NUM_KEYPOINTS
        ))

        frames = np.concatenate([
            frames,
            padding
        ])

    return frames

# ============================================================
# BUILD DATASET
# ============================================================

X = []
y = []

video_extensions = [".mp4", ".MP4"]

cache_dir = Path("cache")
cache_dir.mkdir(exist_ok=True)

for class_name in class_names:

    class_dir = DATA_DIR / class_name

    video_files = []

    for ext in video_extensions:

        video_files.extend(
            class_dir.glob(f"*{ext}")
        )

    print(f"\n{class_name}: {len(video_files)} videos")

    for video_path in tqdm(video_files):

        try:

            # =================================================
            # UNIQUE CACHE NAME
            # =================================================

            safe_name = (
                class_name.replace(" ", "_")
                + "_"
                + video_path.stem.replace(" ", "_")
            )

            cache_path = (
                cache_dir / f"{safe_name}.npy"
            )

            # =================================================
            # LOAD CACHE
            # =================================================

            if cache_path.exists():

                sequence = np.load(cache_path)

            else:

                sequence = load_video(video_path)

                if sequence is not None:
                    np.save(cache_path, sequence)

            if sequence is not None:

                X.append(sequence)
                y.append(class_name)

        except Exception as e:

            print(f"\nError processing:")
            print(video_path)
            print(e)

print("\n✓ Feature extraction completed")

# ============================================================
# CONVERT TO NUMPY
# ============================================================

X = np.array(X)
y = np.array(y)

print("\nDataset Shape:")
print(X.shape)
print(y.shape)

# ============================================================
# ENCODE LABELS
# ============================================================

encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

y_cat = to_categorical(y_encoded)

# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_cat,
    test_size=0.2,
    random_state=42,
    shuffle = True,
)

print("\nTrain Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

# ============================================================
# BUILD MODEL
# ============================================================

model = Sequential([

    Bidirectional(
        LSTM(
            128,
            return_sequences=True
        ),
        input_shape=(SEQ_LEN, NUM_KEYPOINTS)
    ),

    Dropout(0.3),

    Bidirectional(
        LSTM(128)
    ),

    BatchNormalization(),

    Dense(256, activation='relu'),

    Dropout(0.4),

    Dense(128, activation='relu'),

    Dropout(0.3),

    Dense(
        len(class_names),
        activation='softmax'
    )
])

# ============================================================
# COMPILE MODEL
# ============================================================

model.compile(
    optimizer=Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ============================================================
# TRAIN MODEL
# ============================================================

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=8,
    callbacks=[early_stop]
)

# ============================================================
# SAVE MODEL
# ============================================================

model.save("isl_sentence_model.h5")

print("\n✓ Model saved as isl_sentence_model.h5")

# ============================================================
# EVALUATE MODEL
# ============================================================

loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print(f"\nTest Accuracy: {accuracy*100:.2f}%")

I0000 00:00:1778499847.307159 2001231 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1
W0000 00:00:1778499847.338595 2001237 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778499847.388105 2001237 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
E0000 00:00:1778499847.413948 1960129 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-05-11T17:14:16.399197+05:30
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
I0000 00:00:1778499847.533130 2001242 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1
W0000 00:00:1778499847.641202 2001245 inference_feedback_manager.cc:121] Feedback manager requires a model with a single s

✓ MediaPipe Tasks models loaded

Total Classes: 101
['He is going into the room', 'No need to worry dont worry', 'This place is beautiful', 'are you free today', 'are you hiding something', 'bring water for me', 'can i help you', 'can you repeat that please', 'comb your hair', 'congratulations']

He is going into the room: 7 videos


100%|██████████| 7/7 [00:00<00:00, 414.18it/s]



No need to worry dont worry: 1 videos


100%|██████████| 1/1 [00:00<00:00, 663.45it/s]


...

 Total params: 869,477 (3.32 MB)

 Trainable params: 868,965 (3.31 MB)

 Non-trainable params: 512 (2.00 KB)

Epoch 1/100


E0000 00:00:1778499871.660594 1957341 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-05-11T17:26:31.64449+05:30
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1778499878.645250 1950555 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-05-11T17:21:38.60296+05:30
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
